# Test LiquidAI/LFM2.5-2.6B

Loads the locally downloaded model from `./LFM2.5-2.6B` and runs a few prompts.

In [2]:
import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer

MODEL_PATH = "./LFM2.5-2.6B"

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print("device:", device)

/Users/ljx/Documents/work/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: mps


In [3]:
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, dtype=torch.bfloat16).to(device)
model.eval()
print(f"loaded in {time.time() - t0:.1f}s, params: {model.num_parameters() / 1e9:.2f}B")

Loading weights: 100%|██████████| 266/266 [00:00<00:00, 4776.17it/s]


loaded in 6.9s, params: 2.70B


In [4]:
def chat(prompt, max_new_tokens=1024, stream=True):
    input_ids = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        add_generation_prompt=True,
        return_tensors="pt",
        tokenize=True,
        return_dict=True,
    )["input_ids"].to(device)

    streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True) if stream else None
    t0 = time.time()
    with torch.no_grad():
        output = model.generate(
            input_ids,
            do_sample=True,
            temperature=0.1,
            top_k=50,
            repetition_penalty=1.1,
            max_new_tokens=max_new_tokens,
            streamer=streamer,
        )
    new_tokens = output[0, input_ids.shape[1]:]
    elapsed = time.time() - t0
    print(f"\n[{len(new_tokens)} tokens in {elapsed:.1f}s, {len(new_tokens) / elapsed:.1f} tok/s]")
    text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    # The model reasons inside <think>...</think> before answering; return only the answer.
    return text.split("</think>")[-1].strip()

In [5]:
_ = chat("What is C. elegans? Answer in two sentences.")

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


The user asks: "What is C. elegans? Answer in two sentences." So we need to provide a concise answer, exactly two sentences. Must be accurate. Provide definition of Caenorhabditis elegans, its significance as a model organism, etc. Two sentences only.

Potential answer: "C. elegans (Caenorhabditis elegans) is a small, transparent nematode worm that serves as a fundamental model organism in biology due to its simple genome and well-characterized development. It has been extensively studied for its cellular and molecular mechanisms, contributing greatly to our understanding of aging, behavior, and neural circuitry." That's two sentences. Ensure punctuation correct. Count sentences: first ends with period after "development." second ends with period after "circuitry." Yes.

Check if any extra hidden sentence fragments? No. Should be fine.

Thus final answer.</think>C. elegans (Caenorhabditis elegans) is a tiny, transparent nematode worm that serves as a foundational model organism in biol

In [6]:
_ = chat("A road has 3 lanes; one is closed for construction. What fraction of lanes are open?")


Okay, so I need to figure out what fraction of the lanes on a road are open when there's a three-lane road with one lane closed for construction. Let me start by breaking down the problem step by step.

First, the total number of lanes mentioned is 3. Out of these, one lane is closed. That means two lanes are still open. The question is asking for the fraction of lanes that are open. 

Hmm, fractions can sometimes be tricky, but let me recall how they work. A fraction represents a part of a whole. In this case, the whole is all the lanes on the road, which is 3 lanes. The part we're interested in is the number of open lanes, which is 2. So, the fraction should be the number of open lanes divided by the total number of lanes. 

Let me write that down as an equation: Fraction = Open Lanes / Total Lanes. Plugging in the numbers from the problem, that would be 2/3. Wait, is that right? If there are 3 lanes and one is closed, then yes, 2 are open. So 2 divided by 3 equals 2/3. 

But hold o

In [6]:
answer = chat(
    "Extract JSON with keys road, status, reason from this text: "
    "'I-80 eastbound is closed near Truckee due to heavy snow.'",
    stream=False,
)
print(answer)


[366 tokens in 28.7s, 12.8 tok/s]
```json
{
  "road": "I-80",
  "status": "closed",
  "reason": "heavy snow"
}
```
